# ISIC 2019 Silver validation

This notebook validates and normalizes ISIC 2019 Bronze data into Silver: label normalization, image decode/dimension validation, leakage-control groups, and rejected records.

Run `05_setup_tables_and_folders.ipynb` and `10_bronze_ingest.ipynb` before this notebook.

In [ ]:
%run ../_setup_env


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType

from data_platform.datasets.isic_2019 import normalize_labels
from data_platform.setup import build_layout, load_dataset_config, load_yaml_config
from data_platform.spark_io import bronze_image_uri, merge_into

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
bronze_tables = layout["bronze_tables"]
silver_tables = layout["silver_tables"]

print(f"Bronze source metadata table: {bronze_tables['source_metadata']}")
print(f"Bronze image blob table: {bronze_tables['image_blobs']}")
print(f"Silver image inventory table: {silver_tables['image_inventory']}")
print(f"Silver leakage groups table: {silver_tables['leakage_groups']}")
print(f"Silver rejected records table: {silver_tables['rejected_records']}")


## Reconcile Bronze Records

Check that every Bronze image blob has a matching source metadata row before any label or image validation runs. An image blob with no metadata row is otherwise invisible to every downstream check — this notebook only ever looks *from* metadata *to* blobs, so an orphan blob would never be examined at all unless it's explicitly flagged here.

In [ ]:
source_metadata_df = spark.table(bronze_tables["source_metadata"])

image_blob_keys_df = spark.table(bronze_tables["image_blobs"]).select("image_id", "source_split")
metadata_keys_df = source_metadata_df.select("image_id", "source_split")

orphan_blobs_df = image_blob_keys_df.join(metadata_keys_df, on=["image_id", "source_split"], how="left_anti")

orphan_blobs_rejected_df = orphan_blobs_df.select(
    F.col("image_id"),
    bronze_image_uri(bronze_tables["image_blobs"], F.col("source_split"), F.col("image_id")).alias("bronze_uri"),
    F.lit("no matching Bronze source metadata").alias("rejection_reason"),
    F.current_timestamp().alias("rejected_at"),
).cache()
merge_into(spark, orphan_blobs_rejected_df, silver_tables["rejected_records"], ["image_id"])

print(f"Bronze image blobs with no matching source metadata (rejected): {orphan_blobs_rejected_df.count()}")


## Normalize and Validate Labels

Compute `malignancy` and `specific_diagnosis` from Bronze source metadata's diagnosis fields, using `data_platform.datasets.isic_2019.normalize_labels` as a Spark UDF so the mapping logic has one source of truth. This is the cheapest validation gate: rows where neither label resolves are rejected here, before any image bytes are decoded.

In [ ]:
NORMALIZE_LABELS_SCHEMA = MapType(StringType(), StringType())
normalize_labels_udf = F.udf(normalize_labels, NORMALIZE_LABELS_SCHEMA)

labeled_df = (
    source_metadata_df
    .withColumn(
        "labels",
        normalize_labels_udf(F.col("diagnosis_1"), F.col("diagnosis_2"), F.col("diagnosis_3")),
    )
    .withColumn("malignancy", F.col("labels")["malignancy"])
    .withColumn("specific_diagnosis", F.col("labels")["specific_diagnosis"])
    .drop("labels")
    .cache()
)

label_valid_df = labeled_df.where(F.col("malignancy").isNotNull() | F.col("specific_diagnosis").isNotNull()).cache()
unlabeled_df = labeled_df.where(F.col("malignancy").isNull() & F.col("specific_diagnosis").isNull())

unlabeled_rejected_df = unlabeled_df.select(
    F.col("image_id"),
    F.col("source_uri").alias("bronze_uri"),
    F.lit("unmapped or missing label").alias("rejection_reason"),
    F.current_timestamp().alias("rejected_at"),
).cache()
merge_into(spark, unlabeled_rejected_df, silver_tables["rejected_records"], ["image_id"])

print(f"Rows with no resolvable label (rejected): {unlabeled_rejected_df.count()}")
print(f"Rows with at least one resolvable label: {label_valid_df.count()}")


## Validate Images

Decode each label-valid image (left-joined from the Bronze image blob table, so a label-valid row with no matching blob is caught and rejected rather than silently dropped) to check readability and dimensions, batched via `mapInPandas` rather than per-row. Only label-valid rows reach this step, so no compute is spent decoding images that would be rejected anyway.

In [ ]:
import pandas as pd
from pyspark.sql.types import BooleanType, IntegerType, StructField, StructType

from data_platform.validate import decode_image

DECODE_OUTPUT_SCHEMA = StructType(
    [
        StructField("image_id", StringType(), False),
        StructField("source_split", StringType(), False),
        StructField("bronze_uri", StringType(), False),
        StructField("valid", BooleanType(), False),
        StructField("image_width", IntegerType(), True),
        StructField("image_height", IntegerType(), True),
        StructField("image_format", StringType(), True),
        StructField("validation_reason", StringType(), True),
    ]
)


def decode_batch(batches):
    for batch in batches:
        results = []
        for image_id, source_split, bronze_uri, image_bytes in zip(
            batch["image_id"], batch["source_split"], batch["bronze_uri"], batch["image_bytes"]
        ):
            outcome = decode_image(bytes(image_bytes))
            results.append(
                {
                    "image_id": image_id,
                    "source_split": source_split,
                    "bronze_uri": bronze_uri,
                    "valid": outcome["valid"],
                    "image_width": outcome.get("width"),
                    "image_height": outcome.get("height"),
                    "image_format": outcome.get("format"),
                    "validation_reason": outcome.get("reason"),
                }
            )
        yield pd.DataFrame(results)


image_blobs_df = spark.table(bronze_tables["image_blobs"]).select("image_id", "source_split", "image_bytes")
label_valid_with_bytes_df = label_valid_df.join(image_blobs_df, on=["image_id", "source_split"], how="left").cache()

missing_blob_df = label_valid_with_bytes_df.where(F.col("image_bytes").isNull())
matched_with_bytes_df = label_valid_with_bytes_df.where(F.col("image_bytes").isNotNull())

missing_blob_rejected_df = missing_blob_df.select(
    F.col("image_id"),
    F.col("source_uri").alias("bronze_uri"),
    F.lit("missing Bronze image blob").alias("rejection_reason"),
    F.current_timestamp().alias("rejected_at"),
).cache()
merge_into(spark, missing_blob_rejected_df, silver_tables["rejected_records"], ["image_id"])

decoded_df = (
    matched_with_bytes_df
    .select("image_id", "source_split", F.col("source_uri").alias("bronze_uri"), "image_bytes")
    .mapInPandas(decode_batch, schema=DECODE_OUTPUT_SCHEMA)
    .cache()
)

image_invalid_df = decoded_df.where(~F.col("valid"))
image_valid_df = decoded_df.where(F.col("valid"))

image_invalid_rejected_df = image_invalid_df.select(
    F.col("image_id"),
    F.col("bronze_uri"),
    F.col("validation_reason").alias("rejection_reason"),
    F.current_timestamp().alias("rejected_at"),
).cache()
merge_into(spark, image_invalid_rejected_df, silver_tables["rejected_records"], ["image_id"])

print(f"Label-valid rows missing a Bronze image blob (rejected): {missing_blob_rejected_df.count()}")
print(f"Candidate images decoded: {decoded_df.count()}")
print(f"Images failing decode/dimension validation (rejected): {image_invalid_rejected_df.count()}")
print(f"Images passing validation: {image_valid_df.count()}")


## Assign Leakage-Control Groups and Write Image Inventory

Group the accepted images: exact duplicates by `source_checksum` first, then `lesion_id`, then `patient_id`, otherwise a singleton group. Write one row per group to `silver.leakage_groups`, then write the accepted rows (with their assigned `group_id`) to `silver.image_inventory`. Both writes use `MERGE INTO` so reruns are idempotent, matching the pattern already used in `10_bronze_ingest.ipynb`.

In [ ]:
CHECKSUM_DUP_MIN_COUNT = 2

accepted_df = (
    label_valid_df
    .join(image_valid_df, on=["image_id", "source_split"])
    .select(
        F.col("image_id"),
        F.col("bronze_uri"),
        F.col("source_checksum"),
        F.col("image_width"),
        F.col("image_height"),
        F.col("image_format"),
        F.col("malignancy"),
        F.col("specific_diagnosis"),
        F.col("patient_id"),
        F.col("lesion_id"),
    )
)

checksum_counts_df = accepted_df.groupBy("source_checksum").agg(F.count("*").alias("checksum_count"))

# group_source only depends on group_type, so it's a static lookup rather than re-testing
# the checksum/lesion/patient predicates a second time.
GROUP_SOURCE_BY_TYPE = {
    "duplicate": "source_checksum",
    "lesion": "lesion_id",
    "patient": "patient_id",
    "singleton": "image_id",
}
GROUP_SOURCE_MAP = F.create_map(*[F.lit(item) for pair in GROUP_SOURCE_BY_TYPE.items() for item in pair])

grouped_df = (
    accepted_df.join(checksum_counts_df, on="source_checksum")
    .withColumn(
        "group_type",
        F.when(F.col("checksum_count") >= CHECKSUM_DUP_MIN_COUNT, F.lit("duplicate"))
        .when(F.col("lesion_id").isNotNull(), F.lit("lesion"))
        .when(F.col("patient_id").isNotNull(), F.lit("patient"))
        .otherwise(F.lit("singleton")),
    )
    .withColumn("group_source", GROUP_SOURCE_MAP[F.col("group_type")])
    .withColumn(
        # The grouping key's *value* (as opposed to its column name, already resolved above)
        # still needs to branch on group_type, since Spark can't select a column by another
        # column's name without a UDF.
        "group_key_value",
        F.when(F.col("group_type") == "duplicate", F.col("source_checksum"))
        .when(F.col("group_type") == "lesion", F.col("lesion_id"))
        .when(F.col("group_type") == "patient", F.col("patient_id"))
        .otherwise(F.col("image_id")),
    )
    .withColumn("group_id", F.concat(F.col("group_type"), F.lit(":"), F.col("group_key_value")))
    .drop("group_key_value")
    .cache()
)

leakage_groups_df = grouped_df.groupBy("group_id", "group_type", "group_source").agg(
    F.count("*").alias("image_count")
)
merge_into(spark, leakage_groups_df, silver_tables["leakage_groups"], ["group_id"])

image_inventory_df = grouped_df.select(
    F.col("image_id"),
    F.col("bronze_uri"),
    F.col("source_checksum"),
    F.col("image_width"),
    F.col("image_height"),
    F.col("image_format"),
    F.lit("accepted").alias("validation_status"),
    F.lit(None).cast("string").alias("validation_reason"),
    F.col("malignancy"),
    F.col("specific_diagnosis"),
    F.col("patient_id"),
    F.col("lesion_id"),
    F.col("group_id"),
    F.current_timestamp().alias("validated_at"),
)
merge_into(spark, image_inventory_df, silver_tables["image_inventory"], ["image_id"])

print(f"Accepted images written to image_inventory: {image_inventory_df.count()}")
print(f"Leakage-control groups: {leakage_groups_df.count()}")


## Review Outputs

Summary counts, rejection reasons, label distribution, and leakage-group sizes for the latest run.

In [ ]:
print("Rejection reasons:")
display(spark.table(silver_tables["rejected_records"]).groupBy("rejection_reason").count())

image_inventory_review_df = spark.table(silver_tables["image_inventory"]).cache()

print("Validation status:")
display(image_inventory_review_df.groupBy("validation_status").count())

print("Malignancy distribution:")
display(image_inventory_review_df.groupBy("malignancy").count())

print("Specific diagnosis distribution:")
display(image_inventory_review_df.groupBy("specific_diagnosis").count().orderBy(F.desc("count")))

print("Leakage-control group summary:")
display(
    spark.table(silver_tables["leakage_groups"])
    .groupBy("group_type")
    .agg(F.count("*").alias("num_groups"), F.sum("image_count").alias("total_images"))
)
